<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas numpy

In [ ]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os
import json

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected successfully.")

Connected successfully.


In [ ]:
schema = con.execute(
    f"DESCRIBE SELECT * FROM {FACT_MARCH}"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
date_col = "report_date"
client_col = "client_hash_id"
content_col = "content_hash_id"

impressions_col = "gsc_impressions"
clicks_col = "gsc_clicks"

sessions_col = "ga4_sessions"
engaged_sessions_col = "ga4_engaged_sessions"

ga4_available_col = "ga4_data_available"

print("Columns selected successfully.")

Columns selected successfully.


In [ ]:
query = f"""
SELECT
    {client_col} AS client_id,
    {content_col} AS content_id,

    -- March 1-15: information available before prediction
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({clicks_col}, 0)
            ELSE 0
        END
    ) AS clicks_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({sessions_col}, 0)
            ELSE 0
        END
    ) AS sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({engaged_sessions_col}, 0)
            ELSE 0
        END
    ) AS engaged_sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN 1
            ELSE 0
        END
    ) AS ga4_available_days,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({impressions_col}, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS active_days_first_half,

    -- March 16-31: future outcome only
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_second_half

FROM {FACT_MARCH}

WHERE {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'

GROUP BY
    {client_col},
    {content_col}
"""

df = con.execute(query).df()

print("Number of pages:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of pages: 331437


,client_id,content_id,imp_first_half,clicks_first_half,sessions_first_half,engaged_sessions_first_half,ga4_available_days,active_days_first_half,imp_second_half
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,0.0,0.0,0.0,13.0,70.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,0.0,0.0,0.0,9.0,8.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,0.0,0.0,0.0,15.0,680.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,0.0,0.0,0.0,9.0,14.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,0.0,0.0,0.0,14.0,1614.0


In [ ]:
ga4_coverage = (
    df.groupby("ga4_available_days")
    .agg(
        n_pages=("content_id", "size")
    )
    .reset_index()
    .sort_values("ga4_available_days")
)

ga4_coverage["pct_pages"] = (
    ga4_coverage["n_pages"] / len(df) * 100
)

display(ga4_coverage)

print(
    "Pages with NO GA4 data:",
    (df["ga4_available_days"] == 0).sum()
)

print(
    "Pages with SOME GA4 data:",
    (df["ga4_available_days"] > 0).sum()
)

print(
    "Pages with FULL 15 days GA4:",
    (df["ga4_available_days"] == 15).sum()
)


,ga4_available_days,n_pages,pct_pages
0,0.0,280691,84.689096
1,1.0,22744,6.862239
2,2.0,7714,2.327441
3,3.0,4699,1.417766
4,4.0,4193,1.265097
5,5.0,2911,0.878297
6,6.0,2029,0.612183
7,7.0,1392,0.419989
8,8.0,1106,0.333698
9,9.0,912,0.275165


Pages with NO GA4 data: 280691
Pages with SOME GA4 data: 50746
Pages with FULL 15 days GA4: 335


In [ ]:
print(
    "GA4 coverage rate:",
    f"{(df['ga4_available_days'] > 0).mean():.1%}"
)

GA4 coverage rate: 15.3%


In [ ]:
gsc_check = con.execute(f"""
SELECT
    COUNT(DISTINCT content_hash_id) AS total_pages,

    COUNT(DISTINCT CASE
        WHEN gsc_data_available = TRUE
        THEN content_hash_id
    END) AS pages_with_gsc,

    COUNT(DISTINCT CASE
        WHEN gsc_avg_position > 0
        THEN content_hash_id
    END) AS pages_with_position

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
""").df()

display(gsc_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_pages,pages_with_gsc,pages_with_position
0,319759,151981,150675


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
position_features = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE(gsc_sum_position, 0)
            ELSE 0
        END
    ) AS sum_position_first_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    position_features,
    on=["client_id", "content_id"],
    how="left"
)

df["avg_position_first_half"] = np.where(
    df["imp_first_half"] > 0,
    df["sum_position_first_half"] / df["imp_first_half"],
    np.nan
)

print("Rows:", len(df))

display(
    df[
        [
            "content_id",
            "imp_first_half",
            "clicks_first_half",
            "gsc_available_days",
            "avg_position_first_half"
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437


,content_id,imp_first_half,clicks_first_half,gsc_available_days,avg_position_first_half
0,content_d0dff76c889de68f,111.0,0.0,13.0,5.333333
1,content_67741cce996cfafa,38.0,1.0,9.0,4.473684
2,content_2e6360ad20fd7107,219.0,1.0,15.0,4.360731
3,content_ac8663da7484669a,20.0,0.0,9.0,4.150000
4,content_65c50dfe9d87a585,1494.0,0.0,14.0,6.315930
5,content_d49a012dcb924e31,246.0,0.0,15.0,4.808943
6,content_614baf2af4330bd7,413.0,1.0,15.0,4.656174
7,content_4dc944b7d0b65ecc,70.0,0.0,14.0,4.485714
8,content_4a1ca0fa5c177e0c,11.0,0.0,7.0,4.909091
9,content_225dc9235023be5f,279.0,1.0,15.0,10.849462


In [ ]:
second_half_coverage = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days_second_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    second_half_coverage,
    on=["client_id", "content_id"],
    how="left"
)

df["gsc_available_days_second_half"] = (
    df["gsc_available_days_second_half"]
    .fillna(0)
)

print("Done")

display(
    df[
        [
            "content_id",
            "gsc_available_days",
            "gsc_available_days_second_half"
        ]
    ].head(10)
)

Done


,content_id,gsc_available_days,gsc_available_days_second_half
0,content_d0dff76c889de68f,13.0,16.0
1,content_67741cce996cfafa,9.0,7.0
2,content_2e6360ad20fd7107,15.0,16.0
3,content_ac8663da7484669a,9.0,8.0
4,content_65c50dfe9d87a585,14.0,16.0
5,content_d49a012dcb924e31,15.0,16.0
6,content_614baf2af4330bd7,15.0,16.0
7,content_4dc944b7d0b65ecc,14.0,12.0
8,content_4a1ca0fa5c177e0c,7.0,3.0
9,content_225dc9235023be5f,15.0,16.0


In [ ]:
print("First-half GSC coverage:")
print(df["gsc_available_days"].describe())

print("\nSecond-half GSC coverage:")
print(df["gsc_available_days_second_half"].describe())

print(
    "\nPages with full first-half coverage:",
    (df["gsc_available_days"] == 15).sum()
)

print(
    "Pages with full second-half coverage:",
    (df["gsc_available_days_second_half"] == 16).sum()
)

First-half GSC coverage:
count    319759.000000
mean          5.129604
std           6.458892
min           0.000000
25%           0.000000
50%           0.000000
75%          13.000000
max          15.000000
Name: gsc_available_days, dtype: float64

Second-half GSC coverage:
count    331437.000000
mean          5.946301
std           7.112130
min           0.000000
25%           0.000000
50%           1.000000
75%          16.000000
max          16.000000
Name: gsc_available_days_second_half, dtype: float64

Pages with full first-half coverage: 68289
Pages with full second-half coverage: 86722


In [ ]:
clean_df = df[
    (df["gsc_available_days"] == 15) &
    (df["gsc_available_days_second_half"] == 16) &
    (df["imp_first_half"] > 0)
].copy()

print("All pages:", len(df))
print("Clean pages:", len(clean_df))
print(
    "Percentage kept:",
    f"{len(clean_df) / len(df):.1%}"
)

All pages: 331437
Clean pages: 61796
Percentage kept: 18.6%


In [ ]:
# Average daily impressions in each time window
clean_df["avg_daily_imp_first_half"] = (
    clean_df["imp_first_half"] / 15
)

clean_df["avg_daily_imp_second_half"] = (
    clean_df["imp_second_half"] / 16
)

# Percentage change from first half to second half
clean_df["impression_change_pct"] = (
    (
        clean_df["avg_daily_imp_second_half"]
        - clean_df["avg_daily_imp_first_half"]
    )
    / clean_df["avg_daily_imp_first_half"]
) * 100

# Target:
# 1 = impressions declined by more than 20%
# 0 = did not decline by more than 20%
clean_df["is_declining_proxy"] = (
    clean_df["impression_change_pct"] < -20
).astype(int)

# CTR from March 1-15 only
clean_df["ctr_first_half"] = (
    clean_df["clicks_first_half"]
    / clean_df["imp_first_half"]
) * 100

print("Clean pages:", len(clean_df))

print(
    "Declining pages:",
    clean_df["is_declining_proxy"].sum()
)

print(
    "Declining rate:",
    f"{clean_df['is_declining_proxy'].mean():.1%}"
)

display(
    clean_df[
        [
            "content_id",
            "imp_first_half",
            "imp_second_half",
            "impression_change_pct",
            "is_declining_proxy",
            "ctr_first_half",
            "avg_position_first_half"
        ]
    ].head(10)
)

Clean pages: 61796
Declining pages: 20023
Declining rate: 32.4%


,content_id,imp_first_half,imp_second_half,impression_change_pct,is_declining_proxy,ctr_first_half,avg_position_first_half
2,content_2e6360ad20fd7107,219.0,680.0,191.095890,0,0.456621,4.360731
5,content_d49a012dcb924e31,246.0,83.0,-68.368902,1,0.000000,4.808943
6,content_614baf2af4330bd7,413.0,359.0,-18.507869,0,0.242131,4.656174
9,content_225dc9235023be5f,279.0,209.0,-29.771505,1,0.358423,10.849462
17,content_26f5092ee7f70d45,2553.0,1761.0,-35.333431,1,0.000000,7.509205
20,content_cfad137c1b04251b,183.0,255.0,30.635246,0,0.000000,6.852459
22,content_9e7c70abfbae371e,1676.0,760.0,-57.488067,1,0.000000,5.663484
29,content_c0fc3b40ce00a5d7,166.0,169.0,-4.555723,0,0.602410,1.487952
30,content_d79c0df3e1b437fb,167.0,150.0,-15.793413,0,0.000000,1.550898
31,content_f414582a500a5cc0,859.0,949.0,3.572468,0,0.582072,3.047730


In [ ]:
clean_df["volume_bucket"] = pd.qcut(
    clean_df["imp_first_half"].rank(method="first"),
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

volume_table = (
    clean_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        avg_impressions=("imp_first_half", "mean"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

volume_table["decline_rate"] = (
    volume_table["decline_rate"] * 100
)

display(volume_table)

,volume_bucket,n,avg_impressions,decline_rate
0,Low,15449,146.659266,23.347790
1,Medium,15449,423.931322,36.332449
2,High,15449,1086.102725,33.562043
3,Very High,15449,5511.078581,36.364813


### Signal 1 — Search volume

**Verdict: MIXED**

Pages with the lowest first-half search volume had the lowest observed decline rate (23.3%).
The Medium, High, and Very High buckets had higher decline rates of roughly 33.6%–36.4%, but the pattern was not monotonic.

This suggests that search volume alone is not a reliable signal of future decline in this sample.
I therefore should not treat higher volume by itself as evidence that a page is more likely to decline.

In [ ]:
# Keep pages with a valid search position
ctr_position_df = clean_df[
    clean_df["avg_position_first_half"].notna()
    & (clean_df["avg_position_first_half"] > 0)
].copy()

# Create understandable Google position groups
ctr_position_df["position_bucket"] = pd.cut(
    ctr_position_df["avg_position_first_half"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "Top 3",
        "4-10",
        "11-20",
        "21-50",
        "50+"
    ],
    include_lowest=True
)

print("Pages used:", len(ctr_position_df))

display(
    ctr_position_df[
        [
            "content_id",
            "avg_position_first_half",
            "position_bucket",
            "ctr_first_half"
        ]
    ].head(10)
)

Pages used: 61795


,content_id,avg_position_first_half,position_bucket,ctr_first_half
2,content_2e6360ad20fd7107,4.360731,4-10,0.456621
5,content_d49a012dcb924e31,4.808943,4-10,0.000000
6,content_614baf2af4330bd7,4.656174,4-10,0.242131
9,content_225dc9235023be5f,10.849462,11-20,0.358423
17,content_26f5092ee7f70d45,7.509205,4-10,0.000000
20,content_cfad137c1b04251b,6.852459,4-10,0.000000
22,content_9e7c70abfbae371e,5.663484,4-10,0.000000
29,content_c0fc3b40ce00a5d7,1.487952,Top 3,0.602410
30,content_d79c0df3e1b437fb,1.550898,Top 3,0.000000
31,content_f414582a500a5cc0,3.047730,4-10,0.582072


In [ ]:
# Median CTR inside each position group
ctr_position_df["position_ctr_median"] = (
    ctr_position_df
    .groupby("position_bucket", observed=True)["ctr_first_half"]
    .transform("median")
)

# If the median is above zero:
# below median = Low CTR
#
# If the median is zero:
# zero CTR = Low CTR
# positive CTR = High CTR
ctr_position_df["ctr_level"] = np.where(
    ctr_position_df["position_ctr_median"] > 0,
    np.where(
        ctr_position_df["ctr_first_half"]
        < ctr_position_df["position_ctr_median"],
        "Low CTR",
        "High CTR"
    ),
    np.where(
        ctr_position_df["ctr_first_half"] == 0,
        "Low CTR",
        "High CTR"
    )
)

In [ ]:
ctr_position_table = (
    ctr_position_df
    .groupby(
        ["position_bucket", "ctr_level"],
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        avg_ctr=("ctr_first_half", "mean"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

ctr_position_table["decline_rate"] *= 100

display(ctr_position_table)

,position_bucket,ctr_level,n,avg_ctr,decline_rate
0,Top 3,High CTR,3777,0.649353,20.916071
1,Top 3,Low CTR,3776,0.070747,36.599576
2,4-10,High CTR,15296,0.592542,25.202667
3,4-10,Low CTR,15293,0.041607,37.899693
4,11-20,High CTR,5447,0.515441,28.235726
5,11-20,Low CTR,5446,0.000522,30.995226
6,21-50,High CTR,5609,0.300428,47.922981
7,21-50,Low CTR,5771,0.000000,33.962918
8,50+,High CTR,146,0.406912,52.054795
9,50+,Low CTR,1234,0.000000,20.259319


### Signal 2 — CTR vs Position

**Verdict: MIXED**

For pages ranking in the Top 3 and positions 4–10, low CTR was associated with a clearly higher later decline rate.

The relationship was weaker for positions 11–20 and reversed for pages ranking below position 20.

This suggests that low CTR may be a useful review signal when a page already has relatively strong search visibility, but CTR alone should not be used across all ranking positions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

### Baseline rule

Prioritize pages that already rank within the Top 20 but have a lower CTR than other pages with a similar search position.

Among pages that match this rule, pages with more first-half impressions are ranked higher because they represent a larger visible opportunity.

**Reason code:** `GOOD_POSITION_LOW_CTR`

**Action:** `REVIEW_CTR_OPPORTUNITY`

In [ ]:
baseline_df = ctr_position_df.copy()

# Rule condition 1:
# Page is already visible in search (Top 20)
baseline_df["good_position"] = (
    baseline_df["avg_position_first_half"] <= 20
)

# Rule condition 2:
# CTR is low compared with pages in a similar position group
baseline_df["low_ctr_for_position"] = (
    baseline_df["ctr_level"] == "Low CTR"
)

# Page matches our baseline rule only if BOTH conditions are true
baseline_df["rule_match"] = (
    baseline_df["good_position"]
    & baseline_df["low_ctr_for_position"]
)

print(
    "Pages matching rule:",
    baseline_df["rule_match"].sum()
)

print(
    "Percentage matching rule:",
    f"{baseline_df['rule_match'].mean():.1%}"
)

Pages matching rule: 24515
Percentage matching rule: 39.7%


In [ ]:
baseline_df["baseline_score"] = np.where(
    baseline_df["rule_match"],
    baseline_df["imp_first_half"],
    0
)

In [ ]:
baseline_df["reason_code"] = np.where(
    baseline_df["rule_match"],
    "GOOD_POSITION_LOW_CTR",
    ""
)

baseline_df["action"] = np.where(
    baseline_df["rule_match"],
    "REVIEW_CTR_OPPORTUNITY",
    "MONITOR"
)

In [ ]:
baseline_df = (
    baseline_df
    .sort_values(
        [
            "baseline_score",
            "avg_position_first_half"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

display(
    baseline_df[
        [
            "content_id",
            "imp_first_half",
            "avg_position_first_half",
            "ctr_first_half",
            "baseline_score",
            "reason_code",
            "action",
            "is_declining_proxy"
        ]
    ].head(10)
)

,content_id,imp_first_half,avg_position_first_half,ctr_first_half,baseline_score,reason_code,action,is_declining_proxy
0,content_7c6373141eae744a,86860.0,5.953765,0.058715,86860.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
1,content_9c057b66c30a3abb,83772.0,0.105274,0.000000,83772.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
2,content_acbcc847f8996314,83715.0,3.518354,0.158872,83715.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
3,content_34a70fea29d15f24,73639.0,2.948003,0.024444,73639.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
4,content_82e35c4845e6c391,70169.0,17.936539,0.041329,70169.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
5,content_8e1334d6356668e3,58553.0,4.753471,0.001708,58553.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
6,content_65c75874a23fca87,55680.0,6.682004,0.026940,55680.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
7,content_1642f339bd6e7c8d,52378.0,4.651991,0.034366,52378.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
8,content_f6116743b00afc2d,49619.0,9.817671,0.016123,49619.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
9,content_62673eea26c31c17,49386.0,5.729114,0.070870,49386.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1


In [ ]:
base_rate = baseline_df["is_declining_proxy"].mean()

precision_at_10 = baseline_df.head(10)["is_declining_proxy"].mean()
precision_at_100 = baseline_df.head(100)["is_declining_proxy"].mean()

print("Base rate:", f"{base_rate:.1%}")
print("Precision@10:", f"{precision_at_10:.1%}")
print("Precision@100:", f"{precision_at_100:.1%}")

Base rate: 32.4%
Precision@10: 50.0%
Precision@100: 39.0%


### Baseline performance

- Base decline rate: **32.4%**
- Precision@10: **50.0%**
- Precision@100: **39.0%**

The rule improved Precision@100 above the overall decline base rate.
This provides a transparent baseline for the Week 5 model to beat using the same evaluation setup.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.